# TicTacToe

Ismerkedjünk meg a TicTacToe játékkal, és a Minimax alapú megoldás menetével a gyakorlatban!

Elsőként definiáljuk a játék környezetét és szabályait:

### A játéktábla

In [8]:
# Konstansok - nagy betovel jelezni csak konvencio
EMPTY = 0
PLAYER_X = 1  # X
PLAYER_O = 2  # O

# Játéktáblát reprezentáló Board class
class Board:
    def __init__(self):
        self.game_state = [EMPTY] * 9
        self.wins = [
            [0, 1, 2], [3, 4, 5], [6, 7, 8],  # Sorok
            [0, 3, 6], [1, 4, 7], [2, 5, 8],  # Oszlopok
            [0, 4, 8], [2, 4, 6]              # Átlók
        ]

    def copy(self):
        new_board = Board()
        new_board.game_state = self.game_state[:]
        return new_board

    def move(self, player, position):
        if self.game_state[position] != EMPTY:
            raise ValueError("Téves lépés: a mező már foglalt.")
        self.game_state[position] = player

    def is_full(self):
        return all(pos != EMPTY for pos in self.game_state) # all: ha mindenre igaz (tehat ha nincs ures)

    def get_winner(self):
        for win in self.wins:
            a, b, c = win
            if self.game_state[a] == self.game_state[b] == self.game_state[c] != EMPTY:
                return self.game_state[a]
        return EMPTY

    def get_available_moves(self):
        return [i for i in range(9) if self.game_state[i] == EMPTY]

    def display(self):
        symbols = {PLAYER_X: "X", PLAYER_O: "O", EMPTY: " "}
        for i in range(3):
            print(" | ".join(symbols[self.game_state[j]] for j in range(i * 3, (i + 1) * 3)))
            if i < 2:
                print("---------")
        print()

#(teszt)
w = [
    [1,2,3], [4,5,6], [7,8,9]
]
for win in w:
    a,b,c = win
    print(a,b)

1 2
4 5
7 8


A teljesség kedvéért az alábbiakban funkciónként ismerkedjünk meg a működéssel!

A `Board` példány reprezentálja a játéktáblát, amelyen a `game_state` értékei a bal felső sarokból kiindulva, soronként vett mezők értékei; alapból zérók, melyet az `EMPTY` konstans jelez.

In [3]:
b = Board()
b.game_state

[0, 0, 0, 0, 0, 0, 0, 0, 0]

A `copy()` művelet klónozza a példányt: erre később azért lesz szükség, hogy az eredeti táblán való direkt módosítás nélkül tudjunk kiértékelni lehgetséges jövőbeli állapotokat.

A `move(player, position)` segítségével tud a megadott játékos lépni:

In [4]:
b.move(PLAYER_X, 4)
b.game_state

[0, 0, 0, 0, 1, 0, 0, 0, 0]

Kissé előreugorva: a `display()` barátságosabb, $3 \times 3$ megjelenítést produkál:

In [5]:
b.display()

  |   |  
---------
  | X |  
---------
  |   |  



Ne feledkezzünk meg róla, hogy a tömb indexelése zéró-bázisú, tehát a $0$ mezőre lépés a bal felső sarokba, a $4$ épp a tábla közepére lépést jelenti, míg az utolsó mezőre (a játéktábla jobb alsó sarkába) a $8$ index megadásával lehetséges.

Az `is_full()` szimplán ellenőrzi, hogy a játéktábla megtelt-e: ha igen, akkor a játék biztosan véget ért: ehhez a `all(pos != EMPTY for pos in self.game_state)` kifejezést értékeli ki. Mit csinál ez?
- Az `all()` működése egyszerű: végigiterál a paraméterként kapott gyűjteményen, s amennyiben bármely elem `False`, akkor ezzel az értékkel tér vissza; csak akkor ad vissza `True` értéket ha a végigjárt gyűjtemény minden eleme `True`.
- A `pos != EMPTY for pos in self.game_state` list comprehensionnél is alkalmazható úgynevezett generátor kifejezés, amely ebben az esetben minden `game_state` elemre megvizsgálja, hogy nem `EMPTY`-e, s ennek logikai értékével tér vissza.

A játék akkor is véget ér, ha valaki nyert: a `get_winner()` ezt ellenőrzi:

In [ ]:
b.get_winner()

In [ ]:
b.move(PLAYER_X, 3)
b.move(PLAYER_X, 5)
b.get_winner()

A működéséről talán érdemes annyit, hogy a relatíve egyszerű játéktér miatt a nyertes pozíciók elemzése nem ciklusokon alapul, hanem a `wins` tartalmazza az összes nyerő pozíciót:
```
        self.wins = [
            [0, 1, 2], [3, 4, 5], [6, 7, 8],  # Rows
            [0, 3, 6], [1, 4, 7], [2, 5, 8],  # Columns
            [0, 4, 8], [2, 4, 6]              # Diagonals
        ]
```
A `get_winner()` ezt vizsgálja a következőképp:
```
    def get_winner(self):
        """Check if there's a winner."""
        for win in self.wins:
            a, b, c = win
            if self.game_state[a] == self.game_state[b] == self.game_state[c] != EMPTY:
                return self.game_state[a]
        return EMPTY
```
Enumeráláskor az egyes `win` elemek három-elemű listák, ezek elemeit mint `a, b, c` dolgozzuk fel; megvizsgáljuk hogy ezeken a mezőkön azonos, `EMPTY`től eltérő érték szerepel-e, mindezt egy nagy kifejezéssel: ha igen, visszatérünk a nyertes játékos azonosítójával.

### Random ágens

A legegyszerűbb gépi ellenfél az, aki véletlenszerűen lép:

In [ ]:
import random

def random_agent(board, player):
    return random.choice(board.get_available_moves())

Megjegyzés: valójában nem érdekli a random ügynököt hogy `X` vagy `O` jön-e, és hogy melyik kit reprezentál; erre az információra más ügynököknek lehet szüksége, s a közös interfész miatt definiáljuk most így.

### A játék

Kezdjük egy egyszerű játék implementációval, amelyet a továbbiakban tovább bővítünk majd:

In [ ]:
def play_game():
    board = Board()
    current_player = PLAYER_X if input("Ki kezdi a játékot? (X/O): ").strip().upper() == "X" else PLAYER_O

    while True:
        board.display()
        if current_player == PLAYER_X:
            # X vagy Te
            move = int(input("Lépj! (0-8): "))
            try:
                board.move(PLAYER_X, move)
            except ValueError as e:
                print(e)
                continue
        else:
            # O reprezentálja az ágenst
            move = random_agent(board, PLAYER_O)
            board.move(PLAYER_O, move)
            print(f"A {move} mezőre lép az ágens!")

        # Nyert valaki?
        winner = board.get_winner()
        if winner != EMPTY:
            board.display()
            print(f"{'X' if winner == PLAYER_X else 'O'} nyert!")
            break

        # A tábla megtelt?
        if board.is_full():
            board.display()
            print("Döntetlen!")
            break

        # A másik játékos jön
        current_player = PLAYER_X if current_player == PLAYER_O else PLAYER_O


In [ ]:
play_game()

Villámgyorsan kiderül: a random ágens nem túl ütőképes, sok esetben három lépésben legyőzhető. Próbálkozzunk meg más ágens típusokkal is, ehhez viszont előbb bővítsük ki a `play_game()` metódust, hogy támogasson más ágenseket is!

In [ ]:
def play_game(agent="random"):
    board = Board()
    
    # Az elérhető ágensek előre definiálva
    agent_dict = {
        "random": random_agent,
    }
    if agent not in agent_dict:
        raise ValueError(f"Nem létező ágens {agent}! Csak ezeket ismerem: {list(agent_dict.keys())}")

    current_player = PLAYER_X if input("Ki kezdi a játékot? (X/O): ").strip().upper() == "X" else PLAYER_O
    
    while True:
        board.display()
        if current_player == PLAYER_X:
            move = int(input("Lépj! (0-8): "))
            try:
                board.move(PLAYER_X, move)
            except ValueError as e:
                print(e)
                continue
        else:
            # Az ágens lép
            move = agent_dict[agent](board, PLAYER_O)
            board.move(PLAYER_O, move)
            print(f"A {move} mezőre lép az ágens!")

        
        winner = board.get_winner()
        if winner != EMPTY:
            board.display()
            print(f"{'X' if winner == PLAYER_X else 'O'} nyert!")
            break

        if board.is_full():
            board.display()
            print("Döntetlen!")
            break

        current_player = PLAYER_X if current_player == PLAYER_O else PLAYER_O

In [ ]:
#play_game("asd") # Hibára fut

In [ ]:
#play_game()

A későbbiek leegyszerűsítése végett az `agent_dict`et szervezzük ki, s ezzel teljesen moduláris lesz az ágens:

In [ ]:
def play_game(agent="random", agent_dict=None):
    board = Board()
    
    if agent_dict is None:
        raise ValueError(f"agent_dict megadása kötelező")
    if agent not in agent_dict:
        raise ValueError(f"Nem létező ágens {agent}! Csak ezeket ismerem: {list(agent_dict.keys())}")

    current_player = PLAYER_X if input("Ki kezdi a játékot? (X/O): ").strip().upper() == "X" else PLAYER_O
    
    while True:
        board.display()
        if current_player == PLAYER_X:
            move = int(input("Lépj! (0-8): "))
            try:
                board.move(PLAYER_X, move)
            except ValueError as e:
                print(e)
                continue
        else:
            # Az ágens lép
            move = agent_dict[agent](board, PLAYER_O)
            board.move(PLAYER_O, move)
            print(f"A {move} mezőre lép az ágens!")

        
        winner = board.get_winner()
        if winner != EMPTY:
            board.display()
            print(f"{'X' if winner == PLAYER_X else 'O'} nyert!")
            break

        if board.is_full():
            board.display()
            print("Döntetlen!")
            break

        current_player = PLAYER_X if current_player == PLAYER_O else PLAYER_O

In [ ]:
agent_dict = {
    "random": random_agent,
}

In [ ]:
play_game("random", agent_dict)

### Mohó ágens

A mohó ágens nem alkalmaz keresést, mindössze az aktuális állapotot vizsgálja meg, s átnézi a lehetséges lépéseket:
- ha létezik olyan lépés, amellyel nyer, lép;
- ha létezik olyan lépés, amellyel blokkolhatja az ellenfél győzelmét, meglépi.

Ha nincs ilyen lépés, akkor továbbra is random dönt.

In [ ]:
def greedy_agent(board, player):
    opponent = PLAYER_X if player == PLAYER_O else PLAYER_O
    available_moves = board.get_available_moves()

    # Nyerhetek-e egyetlen lépéssel?
    for move in available_moves:
        temp_board = board.copy()
        temp_board.move(player, move)
        if temp_board.get_winner() == player:
            return move

    # Tud-e az ellefelem nyerni egyetlen lépéssel?
    for move in available_moves:
        temp_board = board.copy()
        temp_board.move(opponent, move) # Az ellenfelem lépését játszom el
        if temp_board.get_winner() == opponent:
            return move  # Ha ez a lépés terminál, akkor ezt lépem, hogy blokkoljam a győzelmét

    # Ha egyik sem, akkor random
    return random.choice(available_moves)


In [ ]:
agent_dict = {
    "random": random_agent,
    "greedy": greedy_agent,
}
play_game("greedy", agent_dict)

### Minimax ágens

Az előadáson tanultaknak megfelelően a minimax ágens mélységi kereséssel dolgozik. Bár a TicTacToe esetén nincs túl sok állapot, a lentebbi implementáció mégis alkalmaz két módszert a túlzóan sok vizsgálat elkerülésére:
- Csak előre definiált mélységig lát, mélyebb játékfákat nem jár be
- Alfa-béta nyeséssel elkerüli a garantáltan rosszul teljesítő ágakat


In [ ]:
class MiniMax:
    def __init__(self, max_depth=6):
        self.best_move = None
        self.max_depth = max_depth

    def build_tree(self, board, player):
        self.best_move = None
        self._recursive_build_tree(board, player, 0)
        return self.best_move

    def _recursive_build_tree(self, board, current_player, depth, alpha=-float('inf'), beta=float('inf')):
        if depth > self.max_depth:
            return 0

        winner = board.get_winner()
        if winner == current_player:
            return 1
        elif winner == (PLAYER_X if current_player == PLAYER_O else PLAYER_O):
            return -1

        if board.is_full():
            return 0

        move_list = board.get_available_moves()
        for move in move_list:
            new_board = board.copy()
            new_board.move(current_player, move)

            subalpha = -self._recursive_build_tree(new_board, PLAYER_X if current_player == PLAYER_O else PLAYER_O, depth + 1, -beta, -alpha)

            if subalpha > alpha:
                alpha = subalpha
                if depth == 0:
                    self.best_move = move

            if alpha >= beta:
                break

        return alpha

A belépési pont a `build_tree()` hívás, amely szignatúrája megegyezik a korábban definiált függvényekével; a működéséhez rekurzív függvényhívást alkalmaz. Érdemes rögtön megfigyelni, hogy a rekurzív hívás az alfa értéket adja vissza, a példány saját `best_move` változója tárolja a választott lépést.

A rekurzívan hívott `_recursive_build_tree` első feltétele rögtön a maximális mélység elérését vizsgálja; ha ez megtörtént, akkor visszalép. Hogy ez korrekt legyen, a rekurzív hívásnál `depth + 1` értékkel kell továbblépni.

Ezt követően az aktuális tábla kiértékelése történik: nyilván ez kezdetben nem jut eredményre, ugyanakkor a játékfa bejárása során fog találni termináló lépéseket, melyeket a megfelelő hasznossági értékkel kell ellásson, az előadáson ismertetett módon.

Ha nem terminál az aktuális állapot, akkor a lehetséges lépések közül ki kell választani a racionálisat. Minden egyes lépés kiértékelésre kerül, mely ezzel a rekurzív hívással történik:
```
subalpha = -self._recursive_build_tree(new_board, PLAYER_X if current_player == PLAYER_O else PLAYER_O, depth + 1, -beta, -alpha)
```
Itt a másolt, és adott lépést eljátszott táblán kiértékeljük a másik játékos helyzetét. Az értékek negálásával érjük el azt, hogy ugyanezen logika mentén dolgozhassunk Min és Max esetén is: mindkettő célja a saját `alpha` értékének maximalizálása.

Ezt a maximumkiválasztást végzi a
```
if subalpha > alpha:
    alpha = subalpha
    if depth == 0:
        self.best_move = move
```
blokk, ahol a legjobb lépés és a hozzá tartozó érték tárolásra kerül. Nyilván a lépésnek csak a gyökérelemnél van értelme, ezért feltételes ennek felülírása.

S végül az alfa-béta nyesés kifejezetten elegánsan került megvalósításra az úgynevezett negamax-trükkel: a két negált érték felcserélve kerül átadásra a rekurzív hívásban, vázlatosan mint: `_recursive_build_tree(..., alpha=-beta, beta=-alpha)`

In [ ]:
minimax = MiniMax()
agent_dict = {
    "random": random_agent,
    "greedy": greedy_agent,
    "minimax": lambda board, player: minimax.build_tree(board, player),
}
play_game("minimax", agent_dict)